In [1]:
import os,sys
from tool_server.utils.prompts import tool_planning_model_prompt_one_tool_call
from tool_server.utils.utils import *
# from prompts import pixel_reasoner as first_prompt
import random
from tqdm import tqdm
random.seed(42)
setup_openai_proxy()
gemini_api_key = os.getenv("GEMINI_API_KEY")
assert gemini_api_key, "Please set GEMINI_API_KEY environment variable"

original_data = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/tool_dataset/PixelReasoner-SFT-Data/release.json"
original_data = load_json_file(original_data)


In [11]:
renewed_origin_data = []
for item in original_data:
    message_list = item["message_list"]
    pass_flag = False
    image_dict = {}
    for conv in message_list:
        for content in conv["content"]:
            if "video" in content:
                pass_flag = True
                break
            if "image" in content:
                image_path = content["image"]
                image_dict[image_path] = 1
    if len(image_dict) < 2:
        pass_flag = True
    
    if not pass_flag:
        renewed_origin_data.append(item)

random.shuffle(renewed_origin_data)
len(renewed_origin_data)

2716

In [12]:
renewed_origin_data[0]

{'message_list': [{'role': 'system',
   'content': [{'text': 'You are a helpful assistant.\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"type": "function", "function": {"name": "zoom_in", "description": "Zoom in on the image based on the bounding box coordinates.", "parameters": {"type": "object", "properties": {"bbox_2d": {"type": "array", "description": "normalized coordinates for bounding box of the region you want to zoom in. Values should be within [0.0,1.0].", "items": {"type": "number"}}, "target_image": {"type": "number", "description": "The index of the image to crop. Index from 1 to the number of images. Choose 1 to operate on original image."}}, "required": ["bbox_2d", "target_image"]}}}\n{"type": "function", "function": {"name": "select_frames", "description": "Select frames from a video.", "parameters": {"type": "object", "properties": {"target_frames

In [13]:
k = 400
# candidate = random.sample(renewed_origin_data, k)
candidate_path = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/pixelreasonersft_groundingcrop_candidates.jsonl"
candidate_path2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/pixelreasonersft_crop_candidates.jsonl"
candidate1 = renewed_origin_data[:k]
candidate2 = renewed_origin_data[k:2*k]
# output_path = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/pixelreasoner_sft_.json"
write_jsonl(candidate1, candidate_path)
write_jsonl(candidate2, candidate_path2)

In [16]:
len(candidate1), len(candidate2)

(400, 400)

In [ ]:
# To run this code you need to install the following dependencies:
# pip install google-genai
import base64
import os
from google import genai
from google.genai import types


def gemini_generate(
    first_prompt: str = None,
    question_prompt: str = None,
    api_key: str = os.environ.get("GEMINI_API_KEY"),  
    model = "gemini-2.5-flash",
):
    assert api_key, "Please set GEMINI_API_KEY environment variable"
    assert first_prompt, "first_prompt must be provided"
    assert question_prompt, "question_prompt must be provided"
    
    client = genai.Client(
        api_key=api_key,
    )

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=first_prompt),
            ],
        ),
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=question_prompt),
            ],
        ),
    ]
    generate_content_config = types.GenerateContentConfig(
        thinking_config = types.ThinkingConfig(
            thinking_budget=-1,
        ),
        response_mime_type="text/plain",
    )

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=generate_content_config,
    )
    
    response_text = response.text
    return response_text




In [ ]:
candidate = process_jsonl(candidate_path)

for idx, item in candidate:
    message_list = item["message_list"]
    qid = item["qid"]
    new_idx = f"pixelreasonersft_{qid}"
    question_prompt = f"message_list"
    response_text = gemini_generate(
        first_prompt=first_prompt,
        question_prompt=question_prompt,
    )
    

{'message_list': [{'role': 'system',
   'content': [{'text': 'You are a helpful assistant.\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"type": "function", "function": {"name": "zoom_in", "description": "Zoom in on the image based on the bounding box coordinates.", "parameters": {"type": "object", "properties": {"bbox_2d": {"type": "array", "description": "normalized coordinates for bounding box of the region you want to zoom in. Values should be within [0.0,1.0].", "items": {"type": "number"}}, "target_image": {"type": "number", "description": "The index of the image to crop. Index from 1 to the number of images. Choose 1 to operate on original image."}}, "required": ["bbox_2d", "target_image"]}}}\n{"type": "function", "function": {"name": "select_frames", "description": "Select frames from a video.", "parameters": {"type": "object", "properties": {"target_frames

In [9]:
# gemini_generate(first_prompt=first_prompt, question_prompt=a)
api_key = os.environ.get("GEMINI_API_KEY")
model =  "gemini-2.5-flash"
question_prompt = a 

client = genai.Client(
    api_key=api_key,
)

contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(text=first_prompt),
        ],
    ),
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(text=question_prompt),
        ],
    ),
]
generate_content_config = types.GenerateContentConfig(
    thinking_config = types.ThinkingConfig(
        thinking_budget=-1,
    ),
    response_mime_type="text/plain",
)

response = client.models.generate_content(
    model=model,
    contents=contents,
    config=generate_content_config,
)

In [14]:
# print(response.text)
json_res = response.text.split("```json")[1].split("```")[0]
import json
a = json.loads(json_res)

In [16]:
a[0]

{'role': 'system',
 'content': [{'text': 'You are a visual assistant capable of solving visual reasoning problems. You can rely on your own capabilities or use external tools to assist in solving. \n\nAvailable Tools  \nIn your response, you can use the following tools:  \n\n{\n    "type": "function",\n    "function": {\n        "name": "OCR",\n        "description": "Extracts and localizes text from the given image using OCR. Returns bounding boxes, recognized text, and confidence scores.",\n        "parameters": {\n            "type": "object",\n            "properties": {\n                "image": {\n                    "type": "string",\n                    "description": "The identifier of the image to analyze, e.g., \'img_1\'"\n                }\n            },\n            "required": ["image"]\n        }\n    }\n}\n\n\n{\n    "type": "function",\n    "function": {\n        "name": "Point",\n        "description": "Identify a point in the image based on a natural language descri